# ЛР 04. Бинарная классификация

Задание: импортировать датасет, очистить от пропусков и выбросов, нормализовать/стандартизировать данные, выбрать целевую переменную и признаки, построить модели логистической регрессии, SVM и CART, оценить их качество.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report)

np.random.seed(42)
print("Библиотеки подключены")

## 1. Импорт датасета
Код сам находит первый CSV-файл в папке. Можно явно указать путь в переменной `FILE_PATH`.

In [ ]:
# Укажите путь к файлу, либо оставьте None для авто-поиска первого CSV в текущей папке
FILE_PATH = None

if FILE_PATH is None:
    csv_files = [f for f in os.listdir('.') if f.endswith('.csv')]
    if not csv_files:
        raise FileNotFoundError("CSV-файл не найден в текущей папке")
    FILE_PATH = csv_files[0]

df = pd.read_csv(FILE_PATH)
print(f"Файл: {FILE_PATH}")
print(f"Размер датасета: {df.shape[0]} строк, {df.shape[1]} столбцов")
df.head()

## 2. Обзор и очистка данных
Смотрим структуру, пропуски и статистику.

In [ ]:
df.info()

print("\nПропуски по столбцам:")
print(df.isna().sum())

print("\nОписательная статистика:")
df.describe().round(3)

In [ ]:
# --- 2.1 Пропуски: числовые -> медиана, категориальные -> мода ---
for col in df.columns:
    if df[col].isna().any():
        if pd.api.types.is_numeric_dtype(df[col]):
            df[col] = df[col].fillna(df[col].median())
        else:
            df[col] = df[col].fillna(df[col].mode()[0])
print("Пропусков осталось:", df.isna().sum().sum())

In [ ]:
# --- 2.2 Выбросы: метод межквартильного размаха (IQR) ---
# Применяем только к непрерывным числовым столбцам (много уникальных значений)
num_cols = df.select_dtypes(include=[np.number]).columns
for col in num_cols:
    if df[col].nunique() < 10:      # бинарные/дискретные признаки не трогаем
        continue
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    df = df[(df[col] >= lower) & (df[col] <= upper)]
print(f"Строк после удаления выбросов: {df.shape[0]}")

## 3. Целевая переменная и набор признаков
Целевой столбец выбирается автоматически (бинарный), либо задаётся вручную в `TARGET_COL`.

In [ ]:
# Укажите имя целевого столбца, либо None для авто-подбора
TARGET_COL = None

if TARGET_COL is None:
    # ищем бинарные столбцы: сначала по известным именам, иначе берём последний
    candidates = [c for c in df.columns if df[c].nunique() == 2]
    known = [c for c in candidates if c.lower() in ('target', 'class', 'label', 'is_healthy', 'y')]
    TARGET_COL = known[0] if known else candidates[-1]

print("Целевой столбец:", TARGET_COL)

X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

# Удаляем нечисловые столбцы (имена/идентификаторы)
obj_cols = X.select_dtypes(exclude=[np.number]).columns
X = X.drop(columns=obj_cols)
if len(obj_cols):
    print("Удалены нечисловые столбцы:", list(obj_cols))

print("Признаки:", list(X.columns))
print("Распределение классов:")
print(y.value_counts())

## 4. Стандартизация и разделение выборки

In [ ]:
# Стандартизация числовых признаков (среднее 0, дисперсия 1)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X = pd.DataFrame(X_scaled, columns=X.columns)
print("Признаки после стандартизации (первые 5 строк):")
X.head()

In [ ]:
# Разделение на обучающую (80%) и тестовую (20%) выборки
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Обучающая выборка: {X_train.shape[0]} строк, тестовая: {X_test.shape[0]} строк")

## 5. Подбор гиперпараметров
Перебираем гиперпараметры всех трёх моделей и выбираем наилучшие по метрике F1 с помощью кросс-валидации (`GridSearchCV`): параметр `C` для логистической регрессии и SVM, глубину `max_depth` для дерева решений.

In [ ]:
# Сетки значений гиперпараметров
C_range = [0.01, 0.1, 1, 10, 100]        # C (сила регуляризации)
depth_range = [2, 3, 4, 5, 7, 10]        # max_depth (глубина дерева)

# Логистическая регрессия
lr_gs = GridSearchCV(
    LogisticRegression(max_iter=1000, random_state=42),
    param_grid={"C": C_range},
    cv=5, scoring="f1",
)
lr_gs.fit(X_train, y_train)

# SVM
svm_gs = GridSearchCV(
    SVC(random_state=42),
    param_grid={"C": C_range},
    cv=5, scoring="f1",
)
svm_gs.fit(X_train, y_train)

# Дерево решений (CART)
dt_gs = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid={"max_depth": depth_range},
    cv=5, scoring="f1",
)
dt_gs.fit(X_train, y_train)

print("Логистическая регрессия: лучший C =", lr_gs.best_params_["C"],
      "| F1 (cv) =", round(lr_gs.best_score_, 3))
print("SVM: лучший C =", svm_gs.best_params_["C"],
      "| F1 (cv) =", round(svm_gs.best_score_, 3))
print("CART: лучший max_depth =", dt_gs.best_params_["max_depth"],
      "| F1 (cv) =", round(dt_gs.best_score_, 3))

In [ ]:
# Визуализация: как меняется F1 при разных значениях гиперпараметра
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# C для логистической регрессии и SVM (логарифмическая шкала)
for ax, gs, title in zip(axes[:2], [lr_gs, svm_gs], ["Logistic Regression", "SVM"]):
    ax.plot(C_range, gs.cv_results_["mean_test_score"], marker='o')
    ax.set_xscale('log')
    ax.set_xlabel("C")
    ax.set_ylabel("F1 (cross-validation)")
    ax.set_title(title)
    ax.grid(True)

# max_depth для дерева решений (обычная шкала)
ax = axes[2]
ax.plot(depth_range, dt_gs.cv_results_["mean_test_score"], marker='o')
ax.set_xlabel("max_depth")
ax.set_ylabel("F1 (cross-validation)")
ax.set_title("CART (Decision Tree)")
ax.grid(True)

plt.tight_layout()
plt.show()

## 6. Построение и оценка моделей
Все три модели — с подобранными гиперпараметрами.

In [ ]:
models = {
    "Logistic Regression": lr_gs.best_estimator_,
    "SVM": svm_gs.best_estimator_,
    "CART (Decision Tree)": dt_gs.best_estimator_,
}

predictions = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    predictions[name] = y_pred
    print(f"===== {name} =====")
    print(classification_report(y_test, y_pred, zero_division=0))

In [ ]:
# Сводная таблица метрик
metrics = pd.DataFrame(columns=["Accuracy", "Precision", "Recall", "F1"])
for name, y_pred in predictions.items():
    metrics.loc[name] = [
        accuracy_score(y_test, y_pred),
        precision_score(y_test, y_pred, zero_division=0),
        recall_score(y_test, y_pred, zero_division=0),
        f1_score(y_test, y_pred, zero_division=0),
    ]
metrics = metrics.astype(float).round(3)
print(metrics)

In [ ]:
# Матрицы ошибок
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, y_pred) in zip(axes, predictions.items()):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax)
    ax.set_title(name)
    ax.set_xlabel("Предсказано")
    ax.set_ylabel("Фактически")
plt.tight_layout()
plt.show()

## Выводы
Сравните метрики Accuracy, Precision, Recall и F1. Лучшая модель — с наибольшим F1 (учитывает баланс точности и полноты). Отметьте подобранные гиперпараметры: оптимальные `C` для логистической регрессии и SVM, оптимальную `max_depth` для дерева.